In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from bart_playground import DefaultBART, ParallelTemperingBART, MultiBART, DataGenerator
from bart_playground.samplers import mtmh_proposal_probs
import cProfile
import pstats
import io
import time

In [3]:
# Benchmark settings split into default and MTMH proposal configs
default_proposal_probs = {"grow": 0.4, "prune": 0.4, "change": 0.1, "swap": 0.1}
mtmh_proposal_probs = {"multi_grow": 0.4, "multi_prune": 0.4, "multi_change": 0.1, "multi_swap": 0.1}
generator = DataGenerator(n_samples=1000, n_features=10, noise=0.1, random_seed=42)
X, y = generator.generate(scenario="heteroscedastic")
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

default_kwargs = dict(
    ndpost=800,
    nskip=200,
    n_trees=100,
    proposal_probs=default_proposal_probs,
    random_state=42,
    max_bins=100,
    tol=1,
    quick_decay=False,
    dirichlet_prior=False,
    temperature=1.0,
 )

# PT kwargs template (hard-coded temperatures set in next cell)
pt_kwargs = dict(
    ndpost=800,
    nskip=200,
    n_trees=100,
    proposal_probs=default_proposal_probs,
    random_state=42,
    max_bins=100,
    tol=1,
    quick_decay=False,
    dirichlet_prior=False,
    swap_interval=50,
    swap_sweeps=2,
    post_swap_repair_steps=0,
    store_chain_traces=False,
    store_swap_diagnostics=True,
    # print_swap_diagnostics=True,
    n_temperatures=20, max_temperature=5.0, 
    n_jobs=-1,
    local_move_backend="multiprocessing-pipe",  # or "joblib-loky"
 )

In [4]:
# initialize numba (warmup)
warmup_default = DefaultBART(ndpost=5, nskip=5, n_trees=20, proposal_probs=default_proposal_probs, random_state=7)
_ = warmup_default.fit(X_train[:800], y_train[:800], quietly=True)

warmup_pt = ParallelTemperingBART(
    ndpost=5, nskip=5, n_trees=20, proposal_probs=default_proposal_probs,
    random_state=7, n_temperatures=3, max_temperature=3.0, swap_interval=5
 )
_ = warmup_pt.fit(X_train[:800], y_train[:800], quietly=True)

print("Numba warmup completed.")

Numba warmup completed.


In [5]:
# Quick PT+MTMH smoke test: instantiate PT with MultiSampler and run small fit
pt_mtmh = ParallelTemperingBART(
    ndpost=10, nskip=0, n_trees=5, proposal_probs=mtmh_proposal_probs,
    random_state=7, n_temperatures=3, max_temperature=3.0, swap_interval=5,
    sampler_kind="multi", multi_tries=3
 )
pt_mtmh.fit(X_train[:200], y_train[:200], quietly=True)
print('Chain sampler types:', [type(s).__name__ for s in pt_mtmh.chain_samplers])
print('Trace length (cold chain):', len(pt_mtmh.trace))

Chain sampler types: ['MultiSampler', 'MultiSampler', 'MultiSampler']
Trace length (cold chain): 10


In [6]:
# helper: format cProfile output
def profile_summary(profiler, top_n=20, sort_key="cumtime"):
    s = io.StringIO()
    pstats.Stats(profiler, stream=s).sort_stats(sort_key).print_stats(top_n)
    return s.getvalue()

# helper: compare RMSE across trained models
def compare_rmse(models, X, y):
    import pandas as pd
    rows = []
    for name, model in models.items():
        rmse = root_mean_squared_error(y, model.predict(X))
        rows.append({"model": name, "rmse": rmse})
    return pd.DataFrame(rows).sort_values("rmse", ascending=True).reset_index(drop=True)

# --- DefaultBART fit + profile ---
default_model = DefaultBART(**default_kwargs)
default_profiler = cProfile.Profile()
t0 = time.perf_counter()
default_profiler.enable()
default_model.fit(X_train, y_train)
default_profiler.disable()
default_fit_seconds = time.perf_counter() - t0
default_profile_text = profile_summary(default_profiler, top_n=20)

# --- ParallelTemperingBART (standard PT) fit + profile ---
pt_model = ParallelTemperingBART(**pt_kwargs)
pt_profiler = cProfile.Profile()
t0 = time.perf_counter()
pt_profiler.enable()
pt_model.fit(X_train, y_train)
pt_profiler.disable()
pt_fit_seconds = time.perf_counter() - t0
pt_profile_text = profile_summary(pt_profiler, top_n=20)

# --- ParallelTemperingBART with MultiSampler (PT+MTMH) fit + profile ---
pt_mtmh_kwargs = dict(pt_kwargs)
pt_mtmh_kwargs["proposal_probs"] = mtmh_proposal_probs
pt_mtmh = ParallelTemperingBART(**pt_mtmh_kwargs, sampler_kind="multi")
pt_mtmh_profiler = cProfile.Profile()
t0 = time.perf_counter()
pt_mtmh_profiler.enable()
pt_mtmh.fit(X_train, y_train)
pt_mtmh_profiler.disable()
pt_mtmh_fit_seconds = time.perf_counter() - t0
pt_mtmh_profile_text = profile_summary(pt_mtmh_profiler, top_n=20)

# --- MultiBART (standard MTMH) fit + profile ---
multi_kwargs = dict(default_kwargs)
multi_kwargs["proposal_probs"] = mtmh_proposal_probs
multi_model = MultiBART(**multi_kwargs, multi_tries=10)
multi_profiler = cProfile.Profile()
t0 = time.perf_counter()
multi_profiler.enable()
multi_model.fit(X_train, y_train)
multi_profiler.disable()
multi_fit_seconds = time.perf_counter() - t0
multi_profile_text = profile_summary(multi_profiler, top_n=20)

# RMSE comparison via function (four-way comparison)
rmse_df = compare_rmse(
    {
        "default": default_model,
        "default_pt": pt_model,
        "mtmh": multi_model,
        "mtmh+pt": pt_mtmh,
    },
    X_test,
    y_test,
 )

# speed + rmse summary
import pandas as pd
fit_df = pd.DataFrame({
    "model": ["default", "default_pt", "mtmh+pt", "mtmh"],
    "fit_seconds": [default_fit_seconds, pt_fit_seconds, pt_mtmh_fit_seconds, multi_fit_seconds],
})
fit_df["speedup_vs_default"] = fit_df.loc[0, "fit_seconds"] / fit_df["fit_seconds"]

summary_df = fit_df.merge(rmse_df, on="model", how="left")
print(summary_df)

Iterations: 100%|██████████| 1000/1000 [00:36<00:00, 27.14it/s]


        model  fit_seconds  speedup_vs_default      rmse
0     default     6.904294            1.000000  1.899877
1  default_pt   132.963495            0.051926  1.856165
2     mtmh+pt   200.258091            0.034477  1.886977
3        mtmh    36.864672            0.187288  1.867980
